## 1 setup

In [1]:
%pip install -q pandas pyarrow numpy scipy scikit-learn tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import ast #per convertire liste e diz python in oggetti veri
import json 
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display #per mostrare meglio tabelle e DataFrame
from scipy.sparse import csr_matrix #per matrici sparse
from sklearn.decomposition import TruncatedSVD #per fare decomposizione della matrice utente-prodotto
from sklearn.neighbors import NearestNeighbors #per trovare elementi simili
from tqdm.auto import tqdm

pd.set_option("display.max_columns",80)
pd.set_option("display.max_colwidth",120)

c:\Users\arsid\Desktop\Tesi-ecommerce\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
CATEGORY="AmazonLike_v1"
DATASET_NAME="test_dataset2"
RANDOM_STATE=42
K_VALUES=(5,10,20)
PRIMARY_METRIC="ndcg"
#dobbiamo usare ndcg come metrica, perchè misura quanto bene il modello ordina gli elementi
#nelle prime k posizioni
#sommatoria per le prime k pos di (rilevanza elemento in posizione i/log2(i+1))
#se prodotto fuori dalla topk è uguale a 0
PRIMARY_K=10
#scegli che modelli seguire
RUN_POPULARITY=True
RUN_ITEMKNN=True
RUN_SVD=True

USE_RATING_VALUES=False #stimiamo se utente interagisce col prodotto, non il rating che deve dare
MAX_EVAL_ROWS=None #per primo test su dataset grande, metti None per valutazione completa
SAVE_RANKS=True #salva i rank prodotti dai modelli
SAVE_MODEL_ARTIFACTS=True #salva artefatti,mapping..

ITEMKNN_NEIGHBORS_LIST=[10,50] #lista ridotta per non rendere troppo pesante il primo test multi-categoria
ITEMKNN_N_JOBS=-1 #usa tutte le risorse disponibili

SVD_FACTORS=[4,8,16] #fattori latenti ridotti per primo test su dataset grande
SVD_N_ITER=7 #num iterazioni di svd

DEMO_USERS_N=3 #3 utenti demo
DEMO_TOP_N=10 #mostra 10 raccomandazioni per utente

## 2 percorsi e caricamento dati

In [ ]:
PROJECT_ROOT=Path("..").resolve() if (Path("..")/"data").exists() else Path(".").resolve()
RECOMMENDER_DIR=PROJECT_ROOT/"data"/"processed"/f"{DATASET_NAME}_recommender"
NOTEBOOK04_DIR=RECOMMENDER_DIR/"notebook04_models"
NOTEBOOK04_DIR.mkdir(parents=True,exist_ok=True)

TRAIN_PATH=RECOMMENDER_DIR/"train.parquet"
VALIDATION_EVAL_PATH=RECOMMENDER_DIR/"validation_eval.parquet"
TEST_EVAL_PATH=RECOMMENDER_DIR/"test_eval.parquet"
VALIDATION_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_validation.parquet"
TEST_NEGATIVE_PATH=RECOMMENDER_DIR/"negative_samples_test.parquet"
ITEMS_PATH=RECOMMENDER_DIR/"items.parquet"
USER_MAPPING_PATH=RECOMMENDER_DIR/"user_mapping.parquet"
ITEM_MAPPING_PATH=RECOMMENDER_DIR/"item_mapping.parquet"
SPLIT_SUMMARY_PATH=RECOMMENDER_DIR/"split_summary.json"

print("Project root:",PROJECT_ROOT)
print("Recommender dir:",RECOMMENDER_DIR)
print("Notebook04 dir:",NOTEBOOK04_DIR)


Project root: C:\Users\arsid\Desktop\Tesi-ecommerce\notebooks\test_dataset2
Recommender dir: C:\Users\arsid\Desktop\Tesi-ecommerce\notebooks\test_dataset2\data\processed\test_dataset2_recommender
Notebook04 dir: C:\Users\arsid\Desktop\Tesi-ecommerce\notebooks\test_dataset2\data\processed\test_dataset2_recommender\notebook04_models


In [ ]:
train_df=pd.read_parquet(TRAIN_PATH)
validation_eval_df=pd.read_parquet(VALIDATION_EVAL_PATH)
test_eval_df=pd.read_parquet(TEST_EVAL_PATH)
validation_samples_df=pd.read_parquet(VALIDATION_NEGATIVE_PATH)
test_samples_df=pd.read_parquet(TEST_NEGATIVE_PATH)
items_df=pd.read_parquet(ITEMS_PATH)
user_mapping_df=pd.read_parquet(USER_MAPPING_PATH)
item_mapping_df=pd.read_parquet(ITEM_MAPPING_PATH)

split_summary=json.loads(SPLIT_SUMMARY_PATH.read_text(encoding="utf-8"))

loaded_shapes=pd.DataFrame([
    {"name":"train","rows":len(train_df),"cols":train_df.shape[1]},
    {"name":"validation_eval","rows":len(validation_eval_df),"cols":validation_eval_df.shape[1]},
    {"name":"test_eval","rows":len(test_eval_df),"cols":test_eval_df.shape[1]},
    {"name":"validation_samples","rows":len(validation_samples_df),"cols":validation_samples_df.shape[1]},
    {"name":"test_samples","rows":len(test_samples_df),"cols":test_samples_df.shape[1]},
    {"name":"items","rows":len(items_df),"cols":items_df.shape[1]},
    {"name":"user_mapping","rows":len(user_mapping_df),"cols":user_mapping_df.shape[1]},
    {"name":"item_mapping","rows":len(item_mapping_df),"cols":item_mapping_df.shape[1]},
])

display(loaded_shapes)
display(pd.DataFrame.from_dict(split_summary,orient="index",columns=["value"]).head(30))

,name,rows,cols
0,train,533383,17
1,validation_eval,229415,17
2,test_eval,229395,17
3,validation_samples,229415,8
4,test_samples,229395,8
5,items,19000,17
6,user_mapping,229426,2
7,item_mapping,19000,2


,value
category,AmazonLike_v1
dataset_name,test_dataset2
source_config,user3_item8
positive_rating_threshold,4
min_user_interactions_after,3
min_item_interactions_after,8
n_negatives,100
negative_sampling_mode,mixed_category
same_category_negative_ratio,0.5
random_state,42


In [6]:
def parse_negative_items(value): #prende valore della colonna negative_item_idxs e prova a convertirlo in lista
    if value is None:
        return []
    if isinstance(value,np.ndarray): #se array lo converte in lista python
        value=value.tolist()
    if isinstance(value,(list,tuple,set)): #se già lista, converte ogni elemento in int
        return [int(item) for item in value]
    if isinstance(value,str): #se valore è una stringa, toglie spazi inutili e prova a interpretarla come lista python
        value=value.strip()
        if not value:
            return []
        parsed=ast.literal_eval(value)
        return [int(item) for item in parsed]
    try:
        if pd.isna(value):
            return []
    except (TypeError,ValueError):
        pass
    return [int(value)]

validation_samples_df=validation_samples_df.copy()
test_samples_df=test_samples_df.copy()

validation_samples_df["negative_item_idxs"]=validation_samples_df["negative_item_idxs"].apply(parse_negative_items)
test_samples_df["negative_item_idxs"]=test_samples_df["negative_item_idxs"].apply(parse_negative_items)

if MAX_EVAL_ROWS is not None: #se non nullo, limita righe usate per valutazione
    validation_samples_df=validation_samples_df.head(int(MAX_EVAL_ROWS)).copy()
    test_samples_df=test_samples_df.head(int(MAX_EVAL_ROWS)).copy()

print("Validation samples:",len(validation_samples_df))
print("Test samples:",len(test_samples_df))
display(validation_samples_df.head())

Validation samples: 10000
Test samples: 10000


,split,user_idx,user_id,positive_item_idx,positive_parent_asin,positive_source_category,negative_item_idxs,num_negatives
0,validation,0,AE22236AFRRSMQIKGG7TPTB75QEA,6514,B003ZDO2LM,Electronics,"[6528, 9018, 7947, 1663, 1667, 6210, 9065, 11692, 4199, 10787, 10123, 17156, 9922, 14360, 13938, 6848, 14205, 12535,...",100
1,validation,1,AE222H3FGXWLHRFUMGMS2RR57NDQ,8620,B0054JJ0QW,Electronics,"[9500, 3472, 18447, 108, 2821, 11216, 5422, 2740, 7983, 18783, 6651, 16805, 13464, 445, 11608, 3548, 17867, 9596, 84...",100
2,validation,2,AE223LQXDVA2DHNB7XKAPILBS2DQ,2409,B000Z7AKGC,Electronics,"[6300, 1223, 3559, 15893, 6430, 8971, 4367, 1665, 15335, 18032, 5319, 17590, 1090, 17192, 10396, 7313, 13734, 3807, ...",100
3,validation,3,AE223NW3QAU2KTAY5MAEU6VPBAUA,8210,B004VRQNS6,Electronics,"[13975, 14699, 1135, 2084, 6032, 8377, 8803, 7208, 1591, 16433, 18452, 11349, 11602, 17065, 6775, 1393, 1880, 8353, ...",100
4,validation,4,AE223SBXB2PFTMOITN4V53TLSFSA,18285,B00HJ1NQLS,Electronics,"[13666, 18430, 9327, 4697, 7026, 4014, 16181, 9430, 847, 10157, 14688, 5436, 18876, 3823, 18658, 9035, 1630, 9598, 3...",100


## 3 matrice utente prodotto

In [7]:
#calcoliamo numero totale di utenti e prodotti
n_users=int(max(user_mapping_df["user_idx"].max(),train_df["user_idx"].max())+1) 
n_items=int(max(item_mapping_df["item_idx"].max(),train_df["item_idx"].max())+1)
#estraiamo dal training set due array numpy
train_user_idx=train_df["user_idx"].astype(np.int64).to_numpy()
train_item_idx=train_df["item_idx"].astype(np.int64).to_numpy()

if USE_RATING_VALUES and "rating" in train_df.columns: #se true, allora prendi rating dal dataset
    train_values=train_df["rating"].astype(np.float32).to_numpy()
else:
    train_values=np.ones(len(train_df),dtype=np.float32) #mette 1 se esiste l'interazione, non ci interessa il rating

#costruisci matrice sparsa, righe=utenti, colonne=prodotti, valori=interazioni
train_matrix=csr_matrix((train_values,(train_user_idx,train_item_idx)),shape=(n_users,n_items),dtype=np.float32)
train_matrix.sum_duplicates() #se ci sono duplicati, li somma
train_matrix.eliminate_zeros() #pulisce zero salvati

density=train_matrix.nnz/(n_users*n_items) #calcola densità matrice, nnz=numero celle non nulle

matrix_stats={
    "n_users":int(n_users),
    "n_items":int(n_items),
    "train_interactions":int(train_matrix.nnz),
    "density_pct":float(density*100),
}

display(pd.DataFrame.from_dict(matrix_stats,orient="index",columns=["value"]))

,value
n_users,229426.000000
n_items,19000.000000
train_interactions,533383.000000
density_pct,0.012236


In [8]:
#crea dizionario, per ogni utente crea l'insieme dei prodotti visti nel train
train_seen_by_user=(
    train_df.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#unisce in un unico dataframe le interazioni di train,val,test e crea una tabella con tutte le int note dell'utente
all_interactions_for_seen=pd.concat([train_df,validation_eval_df,test_eval_df],ignore_index=True)
#crea altro dizionario, per ogni utente, insieme di tutti i prodotti visti in train+val+test
all_seen_by_user=(
    all_interactions_for_seen.groupby("user_idx")["item_idx"]
    .apply(lambda values:set(values.astype(int)))
    .to_dict()
)
#crea array con tutti prodotti presenti nel training set
train_item_candidates=np.array(sorted(train_df["item_idx"].astype(int).unique()),dtype=np.int64)

print("Utenti con storico train:",len(train_seen_by_user))
print("Item candidati dal train:",len(train_item_candidates))

Utenti con storico train: 229426
Item candidati dal train: 18996


## 4 funzioni di valutazione

In [9]:
#calcola in che posizione viene messo il prodotto positivo rispetto ai negativi campionati
def rank_positive(user_idx,positive_item_idx,negative_item_idxs,score_fn):
    #costruisce lista dei prodotti candidati da ordinare
    candidate_items=[int(positive_item_idx)]+[int(item) for item in negative_item_idxs]
    #modello assegna punteggio a ogni prodotto candidato
    scores=score_fn(int(user_idx),candidate_items)
    scores=np.asarray(scores,dtype=np.float64) #converte punteggi in array numpy
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    positive_score=float(scores[0]) #punteggio del nostro prodotto positivo
    rank=1
    for item_idx,score in zip(candidate_items[1:],scores[1:]): #scorre solo i prodotti negativi , con [1:] salta il primo
        if score>positive_score: #se prod neg ha punteggio piu alto, peggiora di uno il rank del positivo
            rank+=1
        elif score==positive_score and int(item_idx)<int(positive_item_idx): #vince item con id piu piccolo
            rank+=1
    return int(rank),positive_score #rank=posizione del prodotto tra i negativi+positivo, positive_score=punteggio assegnato

In [10]:
#prende una lista dei rank ottenuti, e i valori di k da valutare
def calculate_topk_metrics(ranks,k_values):
    ranks_array=np.asarray(ranks,dtype=np.float64) #converte lista in array numpy
    rows=[]
    for k in k_values: #scorre tutti i k
        hits=ranks_array<=int(k) #calcola per ogni esempio se il prodotto positivo è entrato in topk
        hitrate=float(np.mean(hits)) if len(ranks_array) else np.nan #calcola % casi in cui il positivo è in topk
        ndcg_values=np.where(hits,1.0/np.log2(ranks_array+1.0),0.0) #calcola valore ndcg@k per ogni esempio
        #where- se hits true calcola la funzione, se è false mette 0
        mrr_values=np.where(hits,1.0/ranks_array,0.0) #calcola mrr#k cioè mean reciprocal rank
        #misura quanto in alto il modello posiziona il prodotto positivo reale, 1/rank
        rows.append({
            "K":int(k),
            "hitrate":hitrate,
            "recall":hitrate,
            "ndcg":float(np.mean(ndcg_values)) if len(ranks_array) else np.nan,
            "mrr":float(np.mean(mrr_values)) if len(ranks_array) else np.nan,
            "n_examples":int(len(ranks_array)),
        })
    return pd.DataFrame(rows)

In [11]:
#valuta un modello di raccomandazione, su val o test
def evaluate_model(samples_df,model_name,score_fn,split_name,k_values=K_VALUES):
    #sampledf df con pos e neg campionati
    ranks=[] #contiene solo rank dei prodotti positivi
    rank_rows=[] #contiene info per ogni esempio
    iterator=samples_df.itertuples(index=False) #trasforma df in iteratore riga per riga
    for row in tqdm(iterator,total=len(samples_df),desc=f"Evaluate {model_name} {split_name}"): #scorre righe in samplesdf
        #restituisce rank la pos nel ranking, e positive_score il punteggio dato
        rank,positive_score=rank_positive(row.user_idx,row.positive_item_idx,row.negative_item_idxs,score_fn)
        ranks.append(rank)
        rank_rows.append({
            "model":model_name,
            "split":split_name,
            "user_idx":int(row.user_idx),
            "positive_item_idx":int(row.positive_item_idx),
            "positive_rank":int(rank),
            "positive_score":float(positive_score),
            "num_negatives":int(len(row.negative_item_idxs)),
            "positive_source_category":getattr(row,"positive_source_category",None),
        })
    metrics_df=calculate_topk_metrics(ranks,k_values) #calcola le metriche topk partendo dai rank
    metrics_df.insert(0,"split",split_name)
    metrics_df.insert(0,"model",model_name)
    ranks_df=pd.DataFrame(rank_rows)
    return metrics_df,ranks_df

## 5 baseline most popular

In [12]:
#costruiamo e valutiamo la baseline

popularity_series=train_df["item_idx"].value_counts() #conta quante volte un prodotto compare nel training set
popularity_scores={int(item_idx):float(count) for item_idx,count in popularity_series.items()} #trasforma la series in diz python

def score_popularity(user_idx,item_idxs): #funzione di scoring, input utente da valutare, e lista di prodotti candidati
    return np.array([popularity_scores.get(int(item_idx),0.0) for item_idx in item_idxs],dtype=np.float64)
    #output un array con un punteggio per ogni prodotto candidato

model_score_functions={} #salva funzioni di scoring dei modelli
validation_metric_frames=[] #salva metriche aggregate dei modelli su val
validation_rank_frames=[] #salva i rank dettagliati dei modelli su val

if RUN_POPULARITY: 
    model_score_functions["MostPopular"]=score_popularity
    #valuta mostpopular sul validation set
    popularity_validation_metrics,popularity_validation_ranks=evaluate_model(validation_samples_df,"MostPopular",score_popularity,"validation")
    
    validation_metric_frames.append(popularity_validation_metrics) #aggiunge i risultati alle liste generali
    validation_rank_frames.append(popularity_validation_ranks)
    display(popularity_validation_metrics)

Evaluate MostPopular validation: 100%|██████████| 10000/10000 [00:00<00:00, 32851.18it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.3442,0.3442,0.247393,0.215530,10000
1,MostPopular,validation,10,0.4720,0.4720,0.288595,0.232460,10000
2,MostPopular,validation,20,0.6297,0.6297,0.328390,0.243343,10000


## 6 itemknn
similarità fra prodotti a partire dagli utenti che li hanno valutati positivamente
per un utente un prodotto riceve punteggio alto se è simile a prodotti già presenti nel suo storico di training

In [13]:
itemknn_artifacts={}
itemknn_neighbor_indices=None #indici dei vicini
itemknn_neighbor_scores=None #punteggi di similarità
empty_int_array=np.array([],dtype=np.int64)
empty_float_array=np.array([],dtype=np.float32)

if RUN_ITEMKNN and n_items>1: 
    itemknn_neighbors_list=sorted(set(int(k) for k in ITEMKNN_NEIGHBORS_LIST if int(k)>0)) #pulisce lista dei valori di k
    #converte ogni k in intero, rimuove valori duplicati e ordina i valori crescenti
    if itemknn_neighbors_list:
        item_user_matrix=train_matrix.T.tocsr() #trasforma la matrice da utenti x prodotti a prodotti x utenti
        max_itemknn_neighbors=int(min(max(itemknn_neighbors_list)+1,n_items)) #prende valore massimo della lista
        itemknn_model=NearestNeighbors(n_neighbors=max_itemknn_neighbors,metric="cosine",algorithm="brute",n_jobs=ITEMKNN_N_JOBS) #crea modello knn
        #metrica coseno per confrontare vettori sparsi e brute per confronto diretto tra prodotti
        itemknn_model.fit(item_user_matrix) #prepara modello sulla matrice
        distances,indices=itemknn_model.kneighbors(item_user_matrix,return_distance=True) #cerca i vicini per ogni prodotto
        similarities=(1.0-distances).astype(np.float32) #trasforma distanze in similarità
        itemknn_neighbor_indices=indices.astype(np.int64) 
        itemknn_neighbor_scores=similarities

        for k in itemknn_neighbors_list: #cicla sui vari k
            model_name=f"ItemKNN_{int(k)}" 
            target_neighbors=int(min(int(k),n_items-1)) #decide quanti vicini usare per quel modello
            current_lookup={}
            for item_idx in range(n_items): #cicla tutti i prodotti, per ogni prodotto costruisce lista dei suoi vicini validi
                current_indices=itemknn_neighbor_indices[item_idx] #prende vicini 
                current_scores=itemknn_neighbor_scores[item_idx] #e suoi punteggi
                keep_mask=(current_indices!=item_idx)&(current_scores>0) #rimuove prodotto stesso e con similarità = 0
                filtered_indices=current_indices[keep_mask][:target_neighbors]
                filtered_scores=current_scores[keep_mask][:target_neighbors]
                current_lookup[int(item_idx)]=(filtered_indices,filtered_scores)
            itemknn_artifacts[model_name]={"k":target_neighbors,"neighbor_lookup":current_lookup}

        print("ItemKNN costruito per K:",itemknn_neighbors_list)
        print("Matrice vicini massima:",itemknn_neighbor_indices.shape)
    else:
        print("ItemKNN saltato: lista vicini vuota")
else:
    print("ItemKNN saltato")

ItemKNN costruito per K: [10, 50]
Matrice vicini massima: (19000, 51)


In [14]:
def make_itemknn_score_fn(current_neighbor_lookup): #funzione che riceve una configurazione di vicini
    def score_itemknn(user_idx,item_idxs):  #calcola punteggi itemknn
        #input utente da valutare e prodotti candidati
        history=train_seen_by_user.get(int(user_idx),set())  #recupera prodotti visti dall'utente nel training
        scores=np.zeros(len(item_idxs),dtype=np.float64)
        if not history:
            return scores
        for pos,item_idx in enumerate(item_idxs): #scorre prodotti candidati
            neigh_items,neigh_scores=current_neighbor_lookup.get(int(item_idx),(empty_int_array,empty_float_array))
            #recupera vicini del prodotto candidato
            total_score=0.0
            for neigh_item,neigh_score in zip(neigh_items,neigh_scores): #scorre insieme prodotti vicini e loro punteggi di similarità
                if int(neigh_item) in history: #se un prodotto vicino è stato visto, aggiunge similarità al punteggio
                    total_score+=float(neigh_score)
            scores[pos]=total_score
        return scores
    return score_itemknn

if RUN_ITEMKNN and itemknn_artifacts:
    for model_name,artifact in itemknn_artifacts.items():
        score_fn=make_itemknn_score_fn(artifact["neighbor_lookup"]) #funzione di scoring
        artifact["score_fn"]=score_fn
        model_score_functions[model_name]=score_fn 
        itemknn_validation_metrics,itemknn_validation_ranks=evaluate_model(validation_samples_df,model_name,score_fn,"validation")
        #valuta questa versione sul validation set
        validation_metric_frames.append(itemknn_validation_metrics)
        validation_rank_frames.append(itemknn_validation_ranks)
        print(model_name,"neighbors",artifact["k"])
        display(itemknn_validation_metrics)
else:
    print("ItemKNN validation saltata")

Evaluate ItemKNN_10 validation: 100%|██████████| 10000/10000 [00:01<00:00, 5354.27it/s]


ItemKNN_10 neighbors 10


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_10,validation,5,0.0727,0.0727,0.054792,0.048977,10000
1,ItemKNN_10,validation,10,0.1090,0.1090,0.066341,0.053634,10000
2,ItemKNN_10,validation,20,0.2013,0.2013,0.089326,0.059761,10000


Evaluate ItemKNN_50 validation: 100%|██████████| 10000/10000 [00:04<00:00, 2451.38it/s]

ItemKNN_50 neighbors 50


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_50,validation,5,0.0920,0.0920,0.070725,0.063753,10000
1,ItemKNN_50,validation,10,0.1297,0.1297,0.082748,0.068618,10000
2,ItemKNN_50,validation,20,0.2176,0.2176,0.104644,0.074458,10000


## 7 matrix factorization con truncated svd
Truncated SVD apprende rappresentazioni latenti di utenti e prodotti a partire dalla matrice di training
Il punteggio utente-prodotto è il prodotto scalare tra embedding utente ed embedding prodotto

In [15]:
svd_artifacts={} #dizionario per salvare modelli SVD addestrati con diversi numeri di componenti
#funzione che da scoring SVD
def make_svd_score_fn(user_embeddings,item_embeddings): #input vettori latenti degli utenti, vettori latenti dei prodotti
    def score_svd(user_idx,item_idxs): #funzione interna che assegna punteggio ai prodotti latenti per un certo utente
        user_vector=user_embeddings[int(user_idx)] #prende vettore latente dell'utente
        item_matrix=item_embeddings[np.asarray(item_idxs,dtype=np.int64)] 
        #prende vettori latenti dei prodotti candidati, converte lista dei prodotti in array di interi
        #per poter indicizzare item_embeddings
        return item_matrix@user_vector #calcola prodotto scalare tra ogni prodotto candidato e l'utente
    return score_svd #restituisce la funzione

def train_svd_model(n_components): #funzione che addestra modello svd
    max_components=max(1,min(n_users,n_items)-1) #max n di componenti
    actual_components=int(min(int(n_components),max_components)) #quante componenti usare davvero
    svd=TruncatedSVD(n_components=actual_components,n_iter=SVD_N_ITER,random_state=RANDOM_STATE) #crea il modello TruncatedSVD
    #ncomponent = n dim latenti, niter = numero iterazioni
    user_embeddings=svd.fit_transform(train_matrix).astype(np.float32) #addestra svd sulla matrice di training e produce gli embedding degli utenti
    #user embeddings ha forma utenti x componenti
    item_embeddings=svd.components_.T.astype(np.float32) #estrae embeddings dei prodotti
    #matrice del tipo prodotti x componenti 
    score_fn=make_svd_score_fn(user_embeddings,item_embeddings) #funzione di scoring per questo specifico modello SVD
    artifact={
        "svd":svd,
        "user_embeddings":user_embeddings,
        "item_embeddings":item_embeddings,
        "explained_variance_ratio":float(svd.explained_variance_ratio_.sum()),
        "requested_components":int(n_components),
        "actual_components":int(actual_components),
        "score_fn":score_fn,
    }
    return artifact

In [16]:
if RUN_SVD and n_users>1 and n_items>1:
    for n_factors in SVD_FACTORS: #scorre tutti i valori latenti che vogliamo provare
        model_name=f"SVD_{int(n_factors)}"
        artifact=train_svd_model(n_factors) #addestra e restituisce dizionario con i vari dati
        svd_artifacts[model_name]=artifact
        model_score_functions[model_name]=artifact["score_fn"] #salva funzione di scoring
        svd_validation_metrics,svd_validation_ranks=evaluate_model(validation_samples_df,model_name,artifact["score_fn"],"validation") 
        #valuta modello svd sul validation set
        validation_metric_frames.append(svd_validation_metrics) 
        validation_rank_frames.append(svd_validation_ranks)
        print(model_name,"explained_variance_ratio",artifact["explained_variance_ratio"])
        display(svd_validation_metrics)
else:
    print("SVD saltato")

Evaluate SVD_4 validation: 100%|██████████| 10000/10000 [00:00<00:00, 38409.83it/s]

SVD_4 explained_variance_ratio 0.010218008421361446


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_4,validation,5,0.3402,0.3402,0.243290,0.211477,10000
1,SVD_4,validation,10,0.4642,0.4642,0.283195,0.227836,10000
2,SVD_4,validation,20,0.6000,0.6000,0.317562,0.237288,10000


Evaluate SVD_8 validation: 100%|██████████| 10000/10000 [00:00<00:00, 41172.13it/s]

SVD_8 explained_variance_ratio 0.018376609310507774


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_8,validation,5,0.3368,0.3368,0.241159,0.209747,10000
1,SVD_8,validation,10,0.4584,0.4584,0.280216,0.225712,10000
2,SVD_8,validation,20,0.5997,0.5997,0.315857,0.235455,10000


Evaluate SVD_16 validation: 100%|██████████| 10000/10000 [00:00<00:00, 40622.76it/s]

SVD_16 explained_variance_ratio 0.030363980680704117


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_16,validation,5,0.3295,0.3295,0.236036,0.205243,10000
1,SVD_16,validation,10,0.4532,0.4532,0.275943,0.221659,10000
2,SVD_16,validation,20,0.5874,0.5874,0.309775,0.230894,10000


## 8 scelta modello su validation

In [17]:
if not validation_metric_frames:
    raise RuntimeError("Nessun modello valutato su validation")

validation_metrics_df=pd.concat(validation_metric_frames,ignore_index=True) #unisce
validation_ranks_df=pd.concat(validation_rank_frames,ignore_index=True) if validation_rank_frames else pd.DataFrame()

target_validation_df=validation_metrics_df[validation_metrics_df["K"]==PRIMARY_K].copy() #tiene solo il k scelto come valore principale
target_validation_df=target_validation_df.sort_values(PRIMARY_METRIC,ascending=False).reset_index(drop=True) #ordina dal migliore al peggiore

best_model_name=str(target_validation_df.loc[0,"model"]) 
best_validation_score=float(target_validation_df.loc[0,PRIMARY_METRIC])

print("Best model:",best_model_name)
print("Best validation score:",best_validation_score)
display(validation_metrics_df)
display(target_validation_df)

Best model: MostPopular
Best validation score: 0.28859502691333394


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,5,0.3442,0.3442,0.247393,0.215530,10000
1,MostPopular,validation,10,0.4720,0.4720,0.288595,0.232460,10000
2,MostPopular,validation,20,0.6297,0.6297,0.328390,0.243343,10000
3,ItemKNN_10,validation,5,0.0727,0.0727,0.054792,0.048977,10000
4,ItemKNN_10,validation,10,0.1090,0.1090,0.066341,0.053634,10000
5,ItemKNN_10,validation,20,0.2013,0.2013,0.089326,0.059761,10000
6,ItemKNN_50,validation,5,0.0920,0.0920,0.070725,0.063753,10000
7,ItemKNN_50,validation,10,0.1297,0.1297,0.082748,0.068618,10000
8,ItemKNN_50,validation,20,0.2176,0.2176,0.104644,0.074458,10000
9,SVD_4,validation,5,0.3402,0.3402,0.243290,0.211477,10000


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,validation,10,0.4720,0.4720,0.288595,0.232460,10000
1,SVD_4,validation,10,0.4642,0.4642,0.283195,0.227836,10000
2,SVD_8,validation,10,0.4584,0.4584,0.280216,0.225712,10000
3,SVD_16,validation,10,0.4532,0.4532,0.275943,0.221659,10000
4,ItemKNN_50,validation,10,0.1297,0.1297,0.082748,0.068618,10000
5,ItemKNN_10,validation,10,0.1090,0.1090,0.066341,0.053634,10000


## 9 valutazione su test

In [18]:
test_metric_frames=[]
test_rank_frames=[]

for model_name,score_fn in model_score_functions.items():
    test_metrics,test_ranks=evaluate_model(test_samples_df,model_name,score_fn,"test")
    test_metric_frames.append(test_metrics)
    test_rank_frames.append(test_ranks)
    display(test_metrics)

test_metrics_df=pd.concat(test_metric_frames,ignore_index=True) if test_metric_frames else pd.DataFrame()
test_ranks_df=pd.concat(test_rank_frames,ignore_index=True) if test_rank_frames else pd.DataFrame()

Evaluate MostPopular test: 100%|██████████| 10000/10000 [00:00<00:00, 35305.89it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,MostPopular,test,5,0.3774,0.3774,0.270279,0.235077,10000
1,MostPopular,test,10,0.5081,0.5081,0.312365,0.252342,10000
2,MostPopular,test,20,0.6534,0.6534,0.349027,0.262366,10000


Evaluate ItemKNN_10 test: 100%|██████████| 10000/10000 [00:01<00:00, 5948.09it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_10,test,5,0.0628,0.0628,0.042889,0.036393,10000
1,ItemKNN_10,test,10,0.1001,0.1001,0.054813,0.041237,10000
2,ItemKNN_10,test,20,0.1825,0.1825,0.075189,0.046590,10000


Evaluate ItemKNN_50 test: 100%|██████████| 10000/10000 [00:03<00:00, 2701.34it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,ItemKNN_50,test,5,0.0737,0.0737,0.051199,0.043835,10000
1,ItemKNN_50,test,10,0.1119,0.1119,0.063440,0.048822,10000
2,ItemKNN_50,test,20,0.1930,0.1930,0.083533,0.054124,10000


... (2 outputs left out)

Evaluate SVD_8 test: 100%|██████████| 10000/10000 [00:00<00:00, 40806.10it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_8,test,5,0.3698,0.3698,0.266472,0.232400,10000
1,SVD_8,test,10,0.4877,0.4877,0.304506,0.248044,10000
2,SVD_8,test,20,0.6259,0.6259,0.339315,0.257527,10000


Evaluate SVD_16 test: 100%|██████████| 10000/10000 [00:00<00:00, 40041.81it/s]


,model,split,K,hitrate,recall,ndcg,mrr,n_examples
0,SVD_16,test,5,0.3551,0.3551,0.253652,0.220202,10000
1,SVD_16,test,10,0.4744,0.4744,0.292098,0.235992,10000
2,SVD_16,test,20,0.6097,0.6097,0.326183,0.245283,10000


In [19]:
all_metrics_df=pd.concat([validation_metrics_df,test_metrics_df],ignore_index=True) #unisce in una unica tabella metriche di val e test

baseline_df=( #tab separata per risultati della baseline most popular
    all_metrics_df[all_metrics_df["model"]=="MostPopular"]
    [["split","K","hitrate","recall","ndcg","mrr"]]
    .rename(columns={
        "hitrate":"hitrate_baseline",
        "recall":"recall_baseline",
        "ndcg":"ndcg_baseline",
        "mrr":"mrr_baseline",
    })
)

comparison_df=all_metrics_df.merge(baseline_df,on=["split","K"],how="left") #unisce tab dei modelli con la tab della baseline

for metric in ["hitrate","recall","ndcg","mrr"]:
    comparison_df[f"{metric}_delta_vs_popularity"]=comparison_df[metric]-comparison_df[f"{metric}_baseline"] #per ogni metrica, calcola quanto è sopra o sotto alla baseline

display(comparison_df.sort_values(["split","K","ndcg"],ascending=[True,True,False]))

,model,split,K,hitrate,recall,ndcg,mrr,n_examples,hitrate_baseline,recall_baseline,ndcg_baseline,mrr_baseline,hitrate_delta_vs_popularity,recall_delta_vs_popularity,ndcg_delta_vs_popularity,mrr_delta_vs_popularity
18,MostPopular,test,5,0.3774,0.3774,0.270279,0.235077,10000,0.3774,0.3774,0.270279,0.235077,0.0000,0.0000,0.000000,0.000000
27,SVD_4,test,5,0.3721,0.3721,0.268084,0.233798,10000,0.3774,0.3774,0.270279,0.235077,-0.0053,-0.0053,-0.002195,-0.001278
30,SVD_8,test,5,0.3698,0.3698,0.266472,0.232400,10000,0.3774,0.3774,0.270279,0.235077,-0.0076,-0.0076,-0.003807,-0.002677
33,SVD_16,test,5,0.3551,0.3551,0.253652,0.220202,10000,0.3774,0.3774,0.270279,0.235077,-0.0223,-0.0223,-0.016627,-0.014875
24,ItemKNN_50,test,5,0.0737,0.0737,0.051199,0.043835,10000,0.3774,0.3774,0.270279,0.235077,-0.3037,-0.3037,-0.219080,-0.191242
21,ItemKNN_10,test,5,0.0628,0.0628,0.042889,0.036393,10000,0.3774,0.3774,0.270279,0.235077,-0.3146,-0.3146,-0.227390,-0.198683
19,MostPopular,test,10,0.5081,0.5081,0.312365,0.252342,10000,0.5081,0.5081,0.312365,0.252342,0.0000,0.0000,0.000000,0.000000
28,SVD_4,test,10,0.4925,0.4925,0.306925,0.249773,10000,0.5081,0.5081,0.312365,0.252342,-0.0156,-0.0156,-0.005440,-0.002569
31,SVD_8,test,10,0.4877,0.4877,0.304506,0.248044,10000,0.5081,0.5081,0.312365,0.252342,-0.0204,-0.0204,-0.007859,-0.004298
34,SVD_16,test,10,0.4744,0.4744,0.292098,0.235992,10000,0.5081,0.5081,0.312365,0.252342,-0.0337,-0.0337,-0.020267,-0.016350


In [20]:
#metriche per categoria del prodotto positivo
#servono a capire se il modello funziona solo su una categoria dominante o anche sulle categorie piccole

def calculate_group_metrics(ranks_df,group_col="positive_source_category",k_values=K_VALUES):
    if ranks_df.empty or group_col not in ranks_df.columns:
        return pd.DataFrame()
    rows=[]
    for (model,split,group_value),group_df in ranks_df.groupby(["model","split",group_col],dropna=False):
        metrics_df=calculate_topk_metrics(group_df["positive_rank"].to_numpy(),k_values)
        metrics_df.insert(0,group_col,group_value)
        metrics_df.insert(0,"split",split)
        metrics_df.insert(0,"model",model)
        rows.append(metrics_df)
    return pd.concat(rows,ignore_index=True) if rows else pd.DataFrame()

all_ranks_df=pd.concat([validation_ranks_df,test_ranks_df],ignore_index=True) if not validation_ranks_df.empty or not test_ranks_df.empty else pd.DataFrame()
category_metrics_df=calculate_group_metrics(all_ranks_df)

if category_metrics_df.empty:
    print("Metriche per categoria non disponibili")
else:
    display(category_metrics_df.sort_values(["split","K","positive_source_category","ndcg"],ascending=[True,True,True,False]))

,model,split,positive_source_category,K,hitrate,recall,ndcg,mrr,n_examples
72,MostPopular,test,Appliances,5,0.284314,0.284314,0.170968,0.133987,204
180,SVD_8,test,Appliances,5,0.259804,0.259804,0.156787,0.123366,204
144,SVD_4,test,Appliances,5,0.259804,0.259804,0.149278,0.113644,204
108,SVD_16,test,Appliances,5,0.225490,0.225490,0.140438,0.112827,204
36,ItemKNN_50,test,Appliances,5,0.112745,0.112745,0.059369,0.042075,204
...,...,...,...,...,...,...,...,...,...
215,SVD_8,validation,Video Games,20,0.734463,0.734463,0.345062,0.235444,531
107,MostPopular,validation,Video Games,20,0.745763,0.745763,0.343870,0.230741,531
179,SVD_4,validation,Video Games,20,0.728814,0.728814,0.343193,0.234500,531
71,ItemKNN_50,validation,Video Games,20,0.318267,0.318267,0.172327,0.132588,531


## 10 salvataggio risultati

In [21]:
METRICS_PATH=NOTEBOOK04_DIR/"notebook04_metrics.csv"
COMPARISON_PATH=NOTEBOOK04_DIR/"notebook04_comparison.csv"
VALIDATION_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_validation_ranks.parquet"
TEST_RANKS_PATH=NOTEBOOK04_DIR/"notebook04_test_ranks.parquet"
SUMMARY_PATH=NOTEBOOK04_DIR/"notebook04_summary.json"
CATEGORY_METRICS_PATH=NOTEBOOK04_DIR/"notebook04_category_metrics.csv"
POPULARITY_SCORES_PATH=NOTEBOOK04_DIR/"popularity_scores.parquet"
ITEMKNN_INDICES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_indices.npy"
ITEMKNN_SCORES_PATH=NOTEBOOK04_DIR/"itemknn_neighbor_scores.npy"
SVD_USER_EMB_PATH=NOTEBOOK04_DIR/"best_svd_user_embeddings.npy"
SVD_ITEM_EMB_PATH=NOTEBOOK04_DIR/"best_svd_item_embeddings.npy"
SVD_INFO_PATH=NOTEBOOK04_DIR/"best_svd_info.json"

popularity_scores_df=pd.DataFrame({
    "item_idx":list(popularity_scores.keys()),
    "popularity_score":list(popularity_scores.values()),
})


In [22]:
############################################################
all_metrics_df.to_csv(METRICS_PATH,index=False) #salva tutte le metriche
comparison_df.to_csv(COMPARISON_PATH,index=False) #salva tab confronto con baseline
if not category_metrics_df.empty:
    category_metrics_df.to_csv(CATEGORY_METRICS_PATH,index=False) #salva metriche per categoria
popularity_scores_df.to_parquet(POPULARITY_SCORES_PATH,index=False) #salva punteggi popolarità

if SAVE_RANKS:
    validation_ranks_df.to_parquet(VALIDATION_RANKS_PATH,index=False)
    test_ranks_df.to_parquet(TEST_RANKS_PATH,index=False)

if SAVE_MODEL_ARTIFACTS and itemknn_neighbor_indices is not None:
    np.save(ITEMKNN_INDICES_PATH,itemknn_neighbor_indices)
    np.save(ITEMKNN_SCORES_PATH,itemknn_neighbor_scores)

best_svd_name=None
if SAVE_MODEL_ARTIFACTS and svd_artifacts:
    svd_candidates=target_validation_df[target_validation_df["model"].isin(svd_artifacts.keys())]
    if not svd_candidates.empty:
        best_svd_name=str(svd_candidates.iloc[0]["model"])
        best_svd_artifact=svd_artifacts[best_svd_name]
        np.save(SVD_USER_EMB_PATH,best_svd_artifact["user_embeddings"])
        np.save(SVD_ITEM_EMB_PATH,best_svd_artifact["item_embeddings"])
        SVD_INFO_PATH.write_text(json.dumps({
            "model":best_svd_name,
            "requested_components":best_svd_artifact["requested_components"],
            "actual_components":best_svd_artifact["actual_components"],
            "explained_variance_ratio":best_svd_artifact["explained_variance_ratio"],
        },indent=2),encoding="utf-8")

summary={
    "category":CATEGORY,
    "dataset_name":DATASET_NAME,
    "primary_metric":PRIMARY_METRIC,
    "primary_k":PRIMARY_K,
    "best_model_validation":best_model_name,
    "best_validation_score":best_validation_score,
    "models_evaluated":list(model_score_functions.keys()),
    "train_rows":int(len(train_df)),
    "validation_samples":int(len(validation_samples_df)),
    "test_samples":int(len(test_samples_df)),
    "n_users":int(n_users),
    "n_items":int(n_items),
    "metrics_path":str(METRICS_PATH),
    "comparison_path":str(COMPARISON_PATH),
    "category_metrics_path":str(CATEGORY_METRICS_PATH) if not category_metrics_df.empty else None,
    "popularity_scores_path":str(POPULARITY_SCORES_PATH),
    "best_svd_name":best_svd_name,
}

SUMMARY_PATH.write_text(json.dumps(summary,indent=2),encoding="utf-8")

display(pd.DataFrame.from_dict(summary,orient="index",columns=["value"]))

,value
category,AmazonLike_v1
dataset_name,test_dataset2
primary_metric,ndcg
primary_k,10
best_model_validation,MostPopular
best_validation_score,0.288595
models_evaluated,"[MostPopular, ItemKNN_10, ItemKNN_50, SVD_4, SVD_8, SVD_16]"
train_rows,533383
validation_samples,10000
test_samples,10000


## 11 raccomandazioni demo

In [23]:
def top_items_for_user(user_idx,score_fn,top_n=DEMO_TOP_N):
    #funzione che restituisce i migliori top_n prodotti per un certo utente
    seen_items=all_seen_by_user.get(int(user_idx),set())
    candidate_items=np.array([item for item in train_item_candidates if int(item) not in seen_items],dtype=np.int64)
    #prodotti candidati
    if len(candidate_items)==0:
        return pd.DataFrame()
    scores=score_fn(int(user_idx),candidate_items) #calcola punteggi di ogni prodotto candidato
    scores=np.asarray(scores,dtype=np.float64)
    scores=np.nan_to_num(scores,nan=-np.inf,posinf=np.inf,neginf=-np.inf)
    order=np.lexsort((candidate_items,-scores))
    top_positions=order[:int(top_n)]
    return pd.DataFrame({
        "user_idx":int(user_idx),
        "rank":np.arange(1,len(top_positions)+1,dtype=np.int64),
        "item_idx":candidate_items[top_positions].astype(int),
        "score":scores[top_positions].astype(float),
    })

def attach_item_metadata(recommendations_df): #funzione che aggiunge info sui prodotti alle raccomandazioni
    if recommendations_df.empty:
        return recommendations_df
    items_for_join=items_df.copy()
    if "item_idx" not in items_for_join.columns:
        items_for_join=items_for_join.merge(item_mapping_df,on="parent_asin",how="left")
    return recommendations_df.merge(items_for_join,on="item_idx",how="left")

In [24]:
demo_user_idxs=(
    test_samples_df["user_idx"]
    .drop_duplicates()
    .head(DEMO_USERS_N) #tiene solo i primi utenti, per demo test
    .astype(int)
    .tolist()
)

best_score_fn=model_score_functions[best_model_name] #recupera funzione migliore, per ora baseline
demo_frames=[]

for user_idx in demo_user_idxs:
    demo_frames.append(top_items_for_user(user_idx,best_score_fn,DEMO_TOP_N)) #per ogni utente genera le raccomandazioni topN

demo_recommendations_df=pd.concat(demo_frames,ignore_index=True) if demo_frames else pd.DataFrame()
demo_recommendations_df=attach_item_metadata(demo_recommendations_df) 

DEMO_RECOMMENDATIONS_PATH=NOTEBOOK04_DIR/"demo_recommendations.csv"
demo_recommendations_df.to_csv(DEMO_RECOMMENDATIONS_PATH,index=False)

preferred_cols=["user_idx","rank","item_idx","parent_asin","source_category","category","title","avg_rating","num_ratings","score"]
display_cols=[col for col in preferred_cols if col in demo_recommendations_df.columns]

display(demo_recommendations_df[display_cols].head(30) if display_cols else demo_recommendations_df.head(30))
print("Demo recommendations path:",DEMO_RECOMMENDATIONS_PATH)

,user_idx,rank,item_idx,parent_asin,source_category,category,title,avg_rating,num_ratings,score
0,0,1,8530,B0052SCU8U,Electronics,Camera & Photo,"Amazon Basics High-Speed HDMI Cable 2-Pack - 6.5 Feet (2 Meters) Supports Ethernet, 3D, 4K and Audio Return",4.7,44359,1519.0
1,0,2,2925,B0019EHU8G,Electronics,Home Audio & Theater,"Mediabridge™ HDMI Cable (6 Feet) Supports 4K@60Hz, High Speed, Hand-Tested, HDMI 2.0 Ready - UHD, 18Gbps, Audio Retu...",4.6,20201,1449.0
2,0,3,15393,B00BWF5U0M,Electronics,Computers,SABRENT 4 Port USB 2.0 Data Hub with Individual LED lit Power Switches [Charging NOT Supported] for Mac & PC (HB-UMLS),4.6,119789,1267.0
3,0,4,6786,B0043T7FXE,Electronics,All Electronics,"Logitech M570 Wireless Trackball Mouse – Ergonomic Design with Sculpted Right-Hand Shape, Compatible with Apple Mac ...",4.6,46483,1227.0
4,0,5,18773,B00IVPU786,Electronics,Electronics,Samsung 32GB up to 48MB/s EVO Class 10 Micro SDHC Card with Adapter (MB-MP32DA/AM),4.5,18234,1149.0
5,0,6,6049,B003NR57BY,Electronics,All Electronics,"Logitech M510 Wireless Computer Mouse – Comfortable Shape with USB Unifying Receiver, with Back/Forward Buttons and ...",4.6,28251,1107.0
6,0,7,18774,B00IVPU7AO,Electronics,Computers,Samsung EVO 64GB 48/MB/s Micro SDXC Memory Card with Adapter up to (MB-MP64DA/AM),4.5,17000,1096.0
7,0,8,3321,B001FWYGJS,Electronics,Computers,"TP-Link N300 Wireless Extender, Wi-Fi Router (TL-WR841N) - 2 x 5dBi High Power Antennas, Supports Access Point, WISP...",4.2,73817,997.0
8,0,9,10110,B006JH8T3S,Electronics,Computers,"Logitech HD Pro Webcam C920, Widescreen Video Calling and Recording, 1080p Camera, Desktop or Laptop Webcam (Discont...",4.6,30686,911.0
9,0,10,6004,B003MTTJOY,Electronics,Computers,"Edimax EW-7811Un 150Mbps 11n Wi-Fi USB Adapter, Nano Size Lets You Plug it and Forget it, Ideal for Raspberry Pi / P...",4.2,10517,891.0


Demo recommendations path: C:\Users\arsid\Desktop\Tesi-ecommerce\notebooks\test_dataset2\data\processed\test_dataset2_recommender\notebook04_models\demo_recommendations.csv
